## Imports

In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

from neuro_fuzzy_toolbox import h_ANFIS, Hybrid_learning_algorithm, EarlyStopping, get_measures

In [2]:
import numpy as np

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Multiclass Classification

## Data

In [4]:
from ucimlrepo import fetch_ucirepo

In [5]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [6]:
X = X.fillna(value=0)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [8]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[115  38  25  25   9]


In [9]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[49 17 11 10  4]


In [10]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [11]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [12]:
y_train.dtype

torch.int64

In [13]:
x_train = x_train.type(torch.float32)
x_test = x_test.type(torch.float32)

In [14]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 4, shuffle = True)
x_trainset = loader.dataset.tensors[0]
y_trainset = loader.dataset.tensors[1]

In [15]:
y_trainset.dtype

torch.int64

## Model & Training

In [22]:
model = h_ANFIS(
    input_size = 13,
    num_mfs = 40,
    outputs = 5,
    rule_reduced = True,
    output_type='multiclass',
    dtype=torch.float32
)

In [23]:
loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.001, 'weight_decay': 0.01}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = EarlyStopping(
    patience=50, 
    delta=0.01
)

In [24]:
trainer = Hybrid_learning_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    validation=0.3,
    early_stopping=early_stopping
)

In [25]:
trainer(model, loader, verbose=True)

Epoch:   1/500 - loss: 1.280581 - validation loss: 1.330154
Epoch:   2/500 - loss: 1.265404 - validation loss: 1.472941
Epoch:   3/500 - loss: 1.282053 - validation loss: 1.399501
Epoch:   4/500 - loss: 1.233361 - validation loss: 1.443484
Epoch:   5/500 - loss: 1.287745 - validation loss: 1.310147
Epoch:   6/500 - loss: 1.255194 - validation loss: 1.403938
Epoch:   7/500 - loss: 1.231241 - validation loss: 1.387805
Epoch:   8/500 - loss: 1.232921 - validation loss: 1.412293
Epoch:   9/500 - loss: 1.213150 - validation loss: 1.447338
Epoch:  10/500 - loss: 1.254197 - validation loss: 1.489284
Epoch:  11/500 - loss: 1.225500 - validation loss: 1.436485
Epoch:  12/500 - loss: 1.212790 - validation loss: 1.418163
Epoch:  13/500 - loss: 1.226302 - validation loss: 1.360582
Epoch:  14/500 - loss: 1.276110 - validation loss: 1.375442
Epoch:  15/500 - loss: 1.285001 - validation loss: 1.451586
Epoch:  16/500 - loss: 1.262080 - validation loss: 1.364327
Epoch:  17/500 - loss: 1.353249 - valida

In [26]:
train_measures = get_measures(model, x_trainset, y_trainset)
for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.5613207547169812
Precision: 0.5600304668172874
Recall: 0.5613207547169812
F1: 0.5484030912041884
Confusion Matrix: [[91 16  6  2  0]
 [11 20  4  1  2]
 [ 4 14  6  0  1]
 [ 4 12  5  1  3]
 [ 1  5  1  1  1]]


In [27]:
test_measures = get_measures(model, x_test, y_test)
for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.4835164835164835
Precision: 0.4518280461676688
Recall: 0.4835164835164835
F1: 0.465364457227544
Confusion Matrix: [[36  8  3  2  0]
 [10  4  3  0  0]
 [ 5  3  3  0  0]
 [ 2  6  0  0  2]
 [ 0  3  0  0  1]]
